*AI-generated draft (Claude, Anthropic) — for review. The counting UI and logic are version-controlled; the worm counts you enter are your own data.*

<span style="font-family: 'Courier New', monospace;">

# 29 · Click-to-count — clear-window validation

Count scale worms by **clicking** on them; counts are written straight into
`validation/clear_window_handcount/handcount_sheet.csv`. Full background is in that
folder's `README.md`.

**Kernel:** `joseph-scaleworm-thesis` (has `ipywidgets` + `ipympl`). If the image
below is not interactive, switch the kernel (top-right) to it and re-run.

**How to use**
1. Run both cells, then type your **initials** in the box.
2. **Left-click each worm** — a yellow ✕ drops and the running **count** (frame title) ticks up.
3. **Right-click** a mark to remove the nearest one; or use **Undo** / **Clear**.
4. **Save & Next ▶** writes the count to the CSV and your exact click positions to
   `clicks/<frame_id>.json` (auditable), then loads the next frame.
5. For dense frames, zoom/pan with the toolbar — but **deselect the zoom tool before
   clicking** (clicks are ignored while zoom or pan is active).
6. If a frame is not a usable front-on view, press **Flag not-front-on** (records a
   note, no count).

Progress is saved after every frame, so you can stop and resume anytime. It opens on
the first frame you haven't counted yet.
</span>

In [3]:
%matplotlib widget
import datetime as dt
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- CONFIG: point this at the handcount session you want to count ----------
# Per-unit detector gates (blind click-count each, then score_unit_gate.py):
BASE = Path("/home/jovyan/scaleworm-student-lab/validation/unit_gates/CAMHDA301-2020")
# BASE = Path("/home/jovyan/scaleworm-student-lab/validation/unit_gates/CAMHDA301-2020")
# BASE = Path("/home/jovyan/scaleworm-student-lab/validation/clear_window_handcount")

SHEET = BASE / "handcount_sheet.csv"
FRAMES = BASE / "frames"
CLICKS = BASE / "clicks"
CLICKS.mkdir(exist_ok=True)

# keep empty cells as "" (not NaN) so we can round-trip the CSV safely
df = pd.read_csv(SHEET, dtype=str, keep_default_na=False)
# A frame is countable if its PNG was extracted. The clear_window sheet also had a
# `frame_ready` gate column; the per-unit gate sheets don't, so treat absent as ready.
ready_idx = [
    i
    for i in df.index
    if df.get("frame_ready", pd.Series("Y", index=df.index)).at[i] != "N"
    and (FRAMES / f"{df.at[i, 'frame_id']}.png").exists()
]
done = sum(1 for i in ready_idx if df.at[i, "worm_count"] != "")
print(f"Counting: {BASE.name}")
print(f"{len(ready_idx)} frames ready to count  ({done} already counted).")
print("Run the next cell, type your initials, then click each worm.")

Counting: CAMHDA301-2020
25 frames ready to count  (0 already counted).
Run the next cell, type your initials, then click each worm.


In [4]:
class WormCounter:
    """Click-to-count worms on validation frames; persists to the hand-count CSV."""

    def __init__(self, df, order):
        self.df = df
        self.order = order  # df indices of ready frames, in display order
        # open on the first not-yet-counted frame
        self.pos = next(
            (k for k, i in enumerate(order) if df.at[i, "worm_count"] == ""), 0
        )
        self.clicks = []
        self.marks = None

        self.initials = widgets.Text(
            value="", description="Initials:", layout=widgets.Layout(width="200px")
        )
        self.status = widgets.HTML()
        self.msg = widgets.Output()

        def mk(desc, style=""):
            return widgets.Button(
                description=desc, button_style=style,
                layout=widgets.Layout(width="auto"),
            )

        self.b_undo = mk("Undo")
        self.b_clear = mk("Clear")
        self.b_prev = mk("◀ Prev")
        self.b_save = mk("Save & Next ▶", "success")
        self.b_flag = mk("Flag not-front-on", "warning")
        self.b_undo.on_click(lambda _: self._undo())
        self.b_clear.on_click(lambda _: self._clear())
        self.b_prev.on_click(lambda _: self._prev())
        self.b_save.on_click(lambda _: self._save_next())
        self.b_flag.on_click(lambda _: self._flag())

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(11, 6.2))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_position = "right"
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)

        controls = widgets.HBox(
            [self.initials, self.b_undo, self.b_clear, self.b_prev, self.b_save, self.b_flag]
        )
        self.box = widgets.VBox([controls, self.status, self.fig.canvas, self.msg])
        self._load()

    def _fid(self):
        return self.df.at[self.order[self.pos], "frame_id"]

    def _clicks_path(self):
        return CLICKS / f"{self._fid()}.json"

    def _load(self):
        fid = self._fid()
        self.ax.clear()
        self.ax.imshow(mpimg.imread(FRAMES / f"{fid}.png"))
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        p = self._clicks_path()
        self.clicks = (
            [tuple(xy) for xy in json.loads(p.read_text())["clicks"]] if p.exists() else []
        )
        self.marks = None
        self._redraw()
        self._status()

    def _redraw(self):
        if self.marks is not None:
            try:
                self.marks.remove()
            except ValueError:
                pass
        if self.clicks:
            xs, ys = zip(*self.clicks)
            self.marks = self.ax.scatter(
                xs, ys, s=90, marker="x", c="#FFD400", linewidths=2, zorder=5
            )
        else:
            self.marks = None
        self.ax.set_title(f"{self._fid()}    —    count: {len(self.clicks)}", fontsize=11)
        self.fig.canvas.draw_idle()

    def _on_click(self, event):
        if event.inaxes != self.ax:
            return
        if self.fig.canvas.toolbar.mode != "":  # zoom/pan active -> ignore
            return
        if event.button == 1:  # left-click: add a worm
            self.clicks.append((event.xdata, event.ydata))
        elif event.button == 3 and self.clicks:  # right-click: remove nearest
            k = min(
                range(len(self.clicks)),
                key=lambda j: (self.clicks[j][0] - event.xdata) ** 2
                + (self.clicks[j][1] - event.ydata) ** 2,
            )
            self.clicks.pop(k)
        self._redraw()

    def _undo(self):
        if self.clicks:
            self.clicks.pop()
            self._redraw()

    def _clear(self):
        self.clicks = []
        self._redraw()

    def _status(self):
        row = self.df.loc[self.order[self.pos]]
        saved = row["worm_count"] or "—"
        note = f" &nbsp; <i>{row['notes']}</i>" if row["notes"] else ""
        self.status.value = (
            f"<b>Frame {self.pos + 1}/{len(self.order)}</b> &nbsp; "
            f"{row['datetime_utc']} &nbsp; {row['camera_unit']} &nbsp; "
            f"saved: <b>{saved}</b> ({row['counter'] or 'uncounted'}){note}"
        )

    def _persist(self, flagged=False):
        who = self.initials.value.strip()
        if not who:
            self.msg.clear_output()
            with self.msg:
                print("⚠ Enter your initials before saving.")
            return False
        i = self.order[self.pos]
        today = dt.date.today().isoformat()
        if flagged:
            self.df.at[i, "worm_count"] = ""
            self.df.at[i, "notes"] = "not front-on"
        else:
            self.df.at[i, "worm_count"] = str(len(self.clicks))
            self._clicks_path().write_text(
                json.dumps(
                    {
                        "frame_id": self._fid(),
                        "counter": who,
                        "date": today,
                        "clicks": [list(c) for c in self.clicks],
                    },
                    indent=1,
                )
            )
        self.df.at[i, "counter"] = who
        self.df.at[i, "date_counted"] = today
        self.df.to_csv(SHEET, index=False)
        return True

    def _advance(self):
        self.msg.clear_output()
        if self.pos < len(self.order) - 1:
            self.pos += 1
            self._load()
        else:
            with self.msg:
                print("✅ All frames counted!")

    def _save_next(self):
        if self._persist():
            self._advance()

    def _flag(self):
        if self._persist(flagged=True):
            self._advance()

    def _prev(self):
        if self.pos > 0:
            self.pos -= 1
            self._load()


app = WormCounter(df, ready_idx)
display(app.box)